# Chapter 11 — Agent Architecture & Agent Evaluation

*Where we are:* a **fixed** RAG pipeline always runs the same steps; an **agent** *chooses* tools
based on the query.

```
query → [ planner → tool calls (schema-validated) → trajectory ] → grounded answer
```

## 54. Why an agent — and why *one controlled* agent

A fixed pipeline can't answer "look up patent US…, then find its classification, then its claims
about X" — that needs conditional tool use. We build **one controlled, rule-based, deterministic**
agent so the trajectory is inspectable and reproducible; the goal is to teach agent *structure*
and *evaluation*, not to build a sprawling multi-agent system.

In [1]:
# === Chapter 11 · standard bootstrap (identical pattern in every notebook) ===
# Runs standalone on a fresh Google Colab VM *or* a local checkout.
import os, sys, subprocess

# After you push this repo to GitHub, put its URL here (one edit works for every chapter):
REPO_URL = "https://github.com/rsalehin/patent-rag-masterclass"   # e.g. "https://github.com/<you>/patent-rag-masterclass"
NEED_OCR = False
IN_COLAB = "google.colab" in sys.modules

if IN_COLAB:
    target = "/content/patent-rag-masterclass"
    if not os.path.isdir(target):
        if REPO_URL:
            subprocess.run(["git", "clone", "--depth", "1", REPO_URL, target], check=True)
        else:
            raise RuntimeError("Set REPO_URL to this repo's GitHub URL (see README.md).")
    os.chdir(target)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", "requirements.txt"], check=True)
    if NEED_OCR:
        subprocess.run(["apt-get", "install", "-y", "-q", "tesseract-ocr"], check=False)

# Ensure the repo root (containing patentrag/) is importable.
for _cand in [os.getcwd()] + [os.path.dirname(os.getcwd())]:
    if os.path.isdir(os.path.join(_cand, "patentrag")):
        if _cand not in sys.path:
            sys.path.insert(0, _cand)
        break

from patentrag import bootstrap as bs
bs.setup_environment(REPO_URL, need_ocr=NEED_OCR)
bs.set_seeds()
_env = bs.environment_report()
print("Chapter 11 bootstrap OK")
print("  Python", _env["python"], "| Colab:", _env["in_colab"], "| CPU cores:", _env["cpu_count"])
print("  torch", _env["torch"], "| CUDA:", _env["cuda_available"], "| tesseract:", _env["tesseract"])

Chapter 11 bootstrap OK
  Python 3.12.10 | Colab: False | CPU cores: 24
  torch 2.12.0.dev20260304+cu130 | CUDA: True | tesseract: True


## 55. Tool schemas (Pydantic / JSON Schema)

Every tool has a strict **Pydantic argument schema** — malformed calls are rejected before
execution. Each tool is read-only and carries a required-permission tag (used by Chapter 12).

In [2]:
import json, pandas as pd
from patentrag.agent import build_backend, SearchPatentsArgs, FetchPatentArgs
registry, docs, by_id, hybrid = build_backend()
print("registered tools:", registry.names())
print("\nJSON schema for search_patents arguments:")
print(json.dumps(SearchPatentsArgs.model_json_schema()["properties"], indent=1))

# schema validation rejects malformed arguments
try:
    FetchPatentArgs(publication_number="not-a-number")
except Exception as e:
    print("\nrejected bad fetch_patent arg ->", type(e).__name__)

registered tools: ['fetch_patent', 'lookup_classification', 'retrieve_passages', 'search_claims', 'search_patents']

JSON schema for search_patents arguments:
{
 "query": {
  "maxLength": 400,
  "minLength": 2,
  "title": "Query",
  "type": "string"
 },
 "top_k": {
  "default": 5,
  "maximum": 50,
  "minimum": 1,
  "title": "Top K",
  "type": "integer"
 }
}

rejected bad fetch_patent arg -> ValidationError


## 56. Agent state & trajectory

The planner maps query features to an ordered tool plan; each step records `(tool, args, result,
error)`. We capture the **full trajectory**.

In [3]:
from patentrag.agent import PatentAgent
from patentrag.generation import generate_answer, MockLLMProvider

def gen(q, passages):
    return generate_answer(q, [by_id[p["chunk_id"]] for p in passages], provider=MockLLMProvider())

agent = PatentAgent(registry, generate_fn=gen)
traj = agent.run("What does patent US9081550B2 disclose in its claims about voice interfaces?")
print("query:", traj.query)
print("tool sequence:", traj.tool_sequence, "\n")
for s in traj.steps:
    head = str(s.result)[:70] if s.result is not None else s.error
    print(f"  → {s.tool:22} args={s.args}")
    print(f"      result: {head}")
print("\nfinal answer citations:", len(traj.final_answer.citations))

C:\Users\rsalehin\AppData\Local\Programs\Python\Python312\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 8474.17it/s]

query: What does patent US9081550B2 disclose in its claims about voice interfaces?
tool sequence: ['fetch_patent', 'search_claims', 'retrieve_passages'] 

  → fetch_patent           args={'publication_number': 'US9081550B2'}
      result: {'publication_number': 'US9081550B2', 'title': 'Adding speech capabili
  → search_claims          args={'query': 'What does patent US9081550B2 disclose in its claims about voice interfaces?', 'top_k': 5}
      result: [{'publication_number': 'US9081550B2', 'claim_number': 28, 'chunk_id':
  → retrieve_passages      args={'query': 'What does patent US9081550B2 disclose in its claims about voice interfaces?', 'top_k': 8}
      result: [{'chunk_id': '20cb637155af', 'publication_number': 'US9081550B2', 'se

final answer citations: 2


In [4]:
# A different query shape yields a different plan (no patent number, mentions "claim").
traj2 = agent.run("Which claims describe approximate nearest neighbor search?")
print("plan for a claim query:", traj2.tool_sequence)

plan for a claim query: ['search_claims', 'retrieve_passages']


## 57–62. Agent evaluation

Agent quality is **more than the final answer** — evaluate tool *selection*, *arguments*, *result
usage*, and the *trajectory*. We define gold trajectories for a small eval set and compute
deterministic metrics.

In [5]:
from patentrag.evaluation import tool_prf, trajectory_match, argument_correctness
# (query, gold tool sequence)
gold = [
    ("Summarize what patent US9081550B2 covers.", ["fetch_patent", "retrieve_passages"]),
    ("What is the CPC classification of US11971885B2?", ["fetch_patent", "lookup_classification", "retrieve_passages"]),
    ("Which claims mention nearest neighbor search?", ["search_claims", "retrieve_passages"]),
    ("Explain retrieval-aware embeddings.", ["retrieve_passages"]),
]
rows = []
for q, gtools in gold:
    pred = agent.run(q).tool_sequence
    prf = tool_prf(pred, gtools)
    rows.append({"query": q[:38], "gold": " → ".join(gtools), "predicted": " → ".join(pred),
                 "tool_F1": round(prf["f1"], 2),
                 "exact_traj": trajectory_match(pred, gtools, "exact"),
                 "subset_ok": trajectory_match(pred, gtools, "subset")})
agent_eval = pd.DataFrame(rows)
agent_eval

,query,gold,predicted,tool_F1,exact_traj,subset_ok
0,Summarize what patent US9081550B2 cove,fetch_patent → retrieve_passages,fetch_patent → retrieve_passages,1.0,True,True
1,What is the CPC classification of US11,fetch_patent → lookup_classification → retriev...,fetch_patent → lookup_classification → retriev...,1.0,True,True
2,Which claims mention nearest neighbor,search_claims → retrieve_passages,search_claims → retrieve_passages,1.0,True,True
3,Explain retrieval-aware embeddings.,retrieve_passages,retrieve_passages,1.0,True,True


In [6]:
# Aggregate agent metrics + argument correctness (did fetch_patent get the right number?)
import numpy as np
tool_f1 = np.mean([r["tool_F1"] for r in rows])
exact_rate = np.mean([r["exact_traj"] for r in rows])
# argument correctness: the pub-number query should fetch exactly that number
t = agent.run("What is the CPC classification of US11971885B2?")
fetch_args = [s.args for s in t.steps if s.tool == "fetch_patent"]
arg_ok = argument_correctness(fetch_args, [{"publication_number": "US11971885B2"}])
metrics = pd.DataFrame([{
    "mean tool-call F1": round(tool_f1, 2),
    "exact-trajectory rate": round(exact_rate, 2),
    "argument correctness": round(arg_ok, 2),
    "avg steps": round(np.mean([len(agent.run(q).steps) for q, _ in gold]), 1),
}])
metrics

,mean tool-call F1,exact-trajectory rate,argument correctness,avg steps
0,1.0,1.0,1.0,2.0


### Trajectory-match variants & failure signatures

- **exact match** — same tools in the same order (strictest)
- **unordered match** — same multiset of tools
- **subset** — the gold tools are all present (extra tools allowed)

Other trajectory pathologies to watch: **unnecessary tool calls**, **loops**, **premature
termination**. Deterministic metrics catch structure; an LLM trajectory-judge (env-gated, like
Ch 10's judge) can assess *reasoning quality* when needed.

**Production implications.** Constrain tools by permission, validate every argument, cap steps to
prevent loops, and log the full trajectory for offline eval. Tool-call F1 and argument
correctness are the leading indicators; final-answer quality is the lagging one.

## Chapter invariants

In [7]:
assert set(registry.names()) >= {"fetch_patent", "search_claims", "retrieve_passages"}
assert traj.tool_sequence[0] == "fetch_patent"           # pub-number query → fetch first
assert "retrieve_passages" in traj.tool_sequence          # always grounds the answer
assert traj.final_answer is not None and traj.final_answer.citations
assert arg_ok == 1.0                                      # right patent number extracted
assert tool_f1 >= 0.8
bs.save_artifact("agent_traces", [{"query": q, "tools": agent.run(q).tool_sequence} for q, _ in gold])
print("All Chapter 11 invariants hold. agent_traces artifact ready.")

All Chapter 11 invariants hold. agent_traces artifact ready.


In [8]:
# === Chapter 11 validation footer ===
import time, platform, sys, importlib.metadata as _md
_pkgs = ['pydantic', 'pandas', 'numpy']
print("Chapter 11 — environment")
print("  Python :", sys.version.split()[0], "on", platform.system(), platform.release())
for _p in _pkgs:
    try: print(f"  {_p:24}: {_md.version(_p)}")
    except Exception: print(f"  {_p:24}: (not installed)")
print()
print("CHAPTER 11 VALIDATION: PASS")

Chapter 11 — environment
  Python : 3.12.10 on Windows 11
  pydantic                : 2.13.3
  pandas                  : 3.0.2
  numpy                   : 2.4.2

CHAPTER 11 VALIDATION: PASS
